In [89]:
from langchain_community.document_loaders import PyMuPDFLoader

loader =  PyMuPDFLoader(file_path="../dataset/half-blood.pdf")

loader = loader.load()

In [92]:
from rich.markdown import Markdown

Markdown(loader[20].page_content)

J.K. Rowling                                                                                                       

HARRY POTTER AND THE HALF-BLOOD PRINCE 21                                                                          

’S END                                                                                                             

Many miles away the chilly mist that had pressed against the Prime Mi- nister’s windows drifted over a dirty river 
that wound between overgrown, rubbish-strewn banks. An immense chimney, relic of a disused mill, reared up, shadowy
and ominous. There was no sound apart from the whisper of the black water and no sign of life apart from a scrawny 
fox that had slunk down the bank to nose hopefully at some old fish-and-chip wrappings in the tall grass. But then,
with a very faint pop, a slim, hooded figure appeared out of thin air on the edge of the river. The fox froze, wary
eyes fixed upon this strange new phenomenon. The figure seemed to take its bearings for a few moments, then set off
with light, quick strides, its long cloak rustling over the grass. With a second and louder pop, another hooded    
figure materialized.                                                                                               
“Wait!” The harsh cry startled the fox, now crouching almost flat in the under- growth. It leapt from its hiding   
place and up the bank. There was a flash of green light, a yelp, and the fox fell back to the ground, dead. The    
second figure turned over the animal with its toe.

In [91]:
import re

def extract_chapter_metadata(text):

    pattern = r'CHAPTER\s+([A-Z0-9]+)\s+([A-Z\'\s]+)'
    match = re.search(pattern, text)

    chapter = None
    title = None

    if match:
        chapter = match.group(1)
        title = match.group(2).strip()

        # remove heading from content
        text = re.sub(pattern, '', text, count=1)

    return text.strip(), chapter, title


for doc in loader:
    
    cleaned_text, chapter, title = extract_chapter_metadata(doc.page_content)

    doc.page_content = cleaned_text
    
    doc.metadata["chapter"] = chapter
    doc.metadata["chapter_title"] = title

In [76]:
for doc in loader:
    doc.page_content = clean_pdf_text(doc.page_content)

In [97]:
loader[4].__dict__

{'id': None,
 'metadata': {'producer': 'Microsoft® Word 2019',
  'creator': 'Microsoft® Word 2019',
  'creationdate': '2021-02-21T13:33:05+01:00',
  'source': '../dataset/half-blood.pdf',
  'file_path': '../dataset/half-blood.pdf',
  'total_pages': 552,
  'format': 'PDF 1.7',
  'title': 'HARRY POTTER AND THE HALF-BLOOD PRINCE',
  'author': 'Bajram Nuredinovski',
  'subject': '',
  'keywords': '',
  'moddate': '2021-02-21T13:52:57+01:00',
  'trapped': '',
  'modDate': "D:20210221135257+01'00'",
  'creationDate': "D:20210221133305+01'00'",
  'page': 4,
  'chapter': 'ONE',
  'chapter_title': 'THE OTHER MINISTER \nI'},
 'page_content': 'J.K. Rowling \n HARRY POTTER AND THE HALF-BLOOD PRINCE \n5 \nt was nearing midnight and the Prime Minister was sitting alone in his\noffice, reading a long memo that was slipping through his brain without lea- \nving the slightest trace of meaning behind. He was waiting for a call from \nthe President of a far distant country, and between wondering when the

In [99]:
import fitz  # PyMuPDF
import re
from collections import Counter


def extract_with_metadata(pdf_path):
    """Extract text blocks with position and font info."""
    doc = fitz.open(pdf_path)
    pages = []

    for page in doc:
        height = page.rect.height
        blocks = []

        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:  # skip images
                continue

            for line in block["lines"]:
                for span in line["spans"]:
                    text = span["text"].strip()
                    if not text:
                        continue

                    blocks.append({
                        "text": text,
                        "y0": span["bbox"][1],      # top position
                        "y1": span["bbox"][3],      # bottom position
                        "font_size": round(span["size"], 1),
                    })

        pages.append({"height": height, "blocks": blocks})

    doc.close()
    return pages


def find_body_font_size(pages):
    """Most common font size = body text."""
    sizes = Counter()
    for page in pages:
        for b in page["blocks"]:
            sizes[b["font_size"]] += len(b["text"])
    return sizes.most_common(1)[0][0]


def is_in_margin(block, page_height, margin=0.08):
    """Is this block in the top/bottom 8% of the page?"""
    return block["y0"] < page_height * margin or block["y1"] > page_height * (1 - margin)


def is_page_number(text):
    """Does this look like a page number?"""
    return bool(re.match(r"^\s*\d{1,4}\s*$", text))


def find_repeated_margin_text(pages, threshold=0.3):
    """Find text that repeats in margins across many pages — these are headers/footers."""
    margin_texts = Counter()

    for page in pages:
        seen_on_page = set()  # count once per page
        for b in page["blocks"]:
            if is_in_margin(b, page["height"], margin=0.10):
                normalized = b["text"].lower().strip()
                if normalized not in seen_on_page and len(normalized) > 1:
                    margin_texts[normalized] += 1
                    seen_on_page.add(normalized)

    min_count = max(2, len(pages) * threshold)
    return {text for text, count in margin_texts.items() if count >= min_count}


def normalize(text):
    """Clean up unicode oddities and whitespace."""
    replacements = {
        "\u2018": "'", "\u2019": "'",    # curly single quotes
        "\u201c": '"', "\u201d": '"',    # curly double quotes
        "\u2013": "-", "\u2014": " - ",  # dashes
        "\u2026": "...",                  # ellipsis
        "\u00a0": " ",                   # non-breaking space
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return re.sub(r" {2,}", " ", text).strip()


def clean_pdf(pdf_path):
    """
    Full pipeline:
      1. Extract blocks with metadata
      2. Find body font size
      3. Find repeated headers/footers
      4. Filter out junk, keep body text
      5. Normalize and return
    """
    # Step 1: Extract
    pages = extract_with_metadata(pdf_path)
    if not pages:
        return ""

    # Step 2: Body font baseline
    body_size = find_body_font_size(pages)
    print(f"Body font size: {body_size}pt")

    # Step 3: Find repeated headers/footers
    repeated = find_repeated_margin_text(pages)
    print(f"Repeated margin text found: {repeated}")

    # Step 4: Filter
    cleaned_parts = []
    removed = {"page_numbers": 0, "headers_footers": 0}

    for page in pages:
        for b in page["blocks"]:
            text = b["text"]
            in_margin = is_in_margin(b, page["height"])

            # Remove page numbers (in margin + looks like a number)
            if in_margin and is_page_number(text):
                removed["page_numbers"] += 1
                continue

            # Remove repeated headers/footers
            if text.lower().strip() in repeated:
                removed["headers_footers"] += 1
                continue

            # Keep everything else
            cleaned_parts.append(text)

    # Step 5: Normalize and join
    raw_text = " ".join(cleaned_parts)
    clean_text = normalize(raw_text)

    # Collapse excessive newlines
    clean_text = re.sub(r"\n{3,}", "\n\n", clean_text)

    # Report
    print(f"\nRemoved {removed['page_numbers']} page numbers")
    print(f"Removed {removed['headers_footers']} header/footer blocks")
    print(f"Final text length: {len(clean_text)} characters")

    return clean_text


# ──────────────────────────────────────────────────────
# RUN IT
# ──────────────────────────────────────────────────────

if __name__ == "__main__":
    text = clean_pdf("../dataset/half-blood.pdf")

    # Preview first 500 characters
    print("\n--- Preview ---")
    print(text[:500])

    # Save cleaned text
    with open("cleaned_output.txt", "w") as f:
        f.write(text)
    print("\nSaved to cleaned_output.txt")

Body font size: 18.0pt
Repeated margin text found: {'j.k. rowling', 'harry potter and the half-blood prince'}

Removed 545 page numbers
Removed 1090 header/footer blocks
Final text length: 980763 characters

--- Preview ---
https://bajrontbooks.com/ Book 6 in the Harry Potter series CHAPTER ONE THE OTHER MINISTER I t was nearing midnight and the Prime Minister was sitting alone in his office, reading a long memo that was slipping through his brain without lea- ving the slightest trace of meaning behind. He was waiting for a call from the President of a far distant country, and between wondering when the wre- tched man would telephone, and trying to suppress unpleasant memories of what had been a very long, tiring, 

Saved to cleaned_output.txt
